# 03 · Objective-1 — gap-fill → features → LightGBM → CV ladder → validation

**BAH 2026 PS3 · Objective-1 (daily surface AQI), the full runnable chain.**

This is the headline Objective-1 pipeline on the offline synthetic data:

1. **Gap-fill** the cloud-holed satellite columns — `aqi_india.fusion.gapfill.fill_gaps`.
2. **Feature matrix** — `aqi_india.features.feature_matrix.build_feature_matrix` (H3-keyed,
   physics-guided channels, met, static covars, temporal encodings, fire context).
3. **Train** the per-pollutant `MultiPollutantLGBM` (monotone-AOD constraint).
4. **CV ladder** — random vs leave-station-out vs spatiotemporal-blocked, to expose leakage.
5. **Validation** — Taylor diagram, 1:1 hexbin, residual map and a metrics table.

In [ ]:
import sys, pathlib
# Make the src/ layout importable when running from the notebooks/ folder
# without an editable install. If aqi_india is already installed this is a no-op.
_repo = pathlib.Path.cwd()
for _ in range(4):
    if (_repo / 'src' / 'aqi_india').is_dir():
        sys.path.insert(0, str(_repo / 'src'))
        break
    _repo = _repo.parent
import aqi_india
print('aqi_india', aqi_india.__version__)

## 1. Build the synthetic dataset and gap-fill the cube

The satellite columns have 30–50% cloud gaps per day. `fill_gaps` runs the layered cascade
(DINEOF → IDW → reanalysis-prior fallback) and reports the gap fraction before/after,
carrying a `<var>_uncertainty` channel. Reanalysis-style met vars are gap-free priors.

In [ ]:
import numpy as np, pandas as pd
from aqi_india.sim import synthetic as sim
from aqi_india.fusion.gapfill import fill_gaps

SEED = 42
grid_raw = sim.make_grid('2023-10-01', n_days=60, res=0.25, seed=SEED)
fires = sim.make_fires(grid_raw, season='oct_nov', seed=SEED)
grid_raw = sim.inject_fire_hcho(grid_raw, fires)
stations = sim.make_stations(grid_raw, n=120, seed=SEED)

grid = fill_gaps(grid_raw, method='auto')
reports = pd.DataFrame(grid.attrs['gapfill_reports'])
print('overall gap fraction  before: %.3f  after: %.3f'
      % (grid.attrs['gapfill_gap_before'], grid.attrs['gapfill_gap_after']))
reports[['variable', 'gap_before', 'gap_after', 'method']]

## 2. Build the feature matrix

`build_feature_matrix(grid, stations, fires)` samples the gap-filled cube at each station's
nearest grid node (cKDTree), derives the physics channels (`aod_pbl`, `aod_dry`, `fnr`,
`wind_speed/dir`), attaches static covariates and day-of-year encodings, merges fire context
per `(h3_res7, time)`, and adds short lags — one tidy row per station-day, keyed on the
universal H3 cell. `feature_columns` returns just the predictor columns.

In [ ]:
from aqi_india.features.feature_matrix import build_feature_matrix, feature_columns

matrix = build_feature_matrix(grid, stations, fires, add_lags=True)
feat_cols = feature_columns(matrix)
print('matrix:', matrix.shape, '| predictors:', len(feat_cols))
print('predictor columns:', feat_cols)
matrix.head()

## 3. Train the MultiPollutantLGBM model

`MultiPollutantLGBM` fits one LightGBM regressor per surface pollutant, with a **monotone-up**
constraint on every AOD-derived feature (more aerosol never lowers predicted PM) and optional
quantile heads for prediction intervals. We fit it directly here on the full matrix.

In [ ]:
from aqi_india.models.lightgbm_model import MultiPollutantLGBM, LGBMParams, SURFACE_POLLUTANTS

labels = {p: matrix[p].to_numpy(float) for p in SURFACE_POLLUTANTS if p in matrix}
params = LGBMParams(n_estimators=400, learning_rate=0.05, seed=SEED)  # light for the demo
model = MultiPollutantLGBM(pollutants=tuple(labels), params=params)
model.fit(matrix[feat_cols], labels, fit_quantiles=True)
print('fitted pollutants:', model.fitted_pollutants)

### Feature importances (driver attribution)

Gain-based importances per pollutant. The AOD / `aod_pbl` channels should dominate PM2.5/PM10,
consistent with the AOD→PM physics the simulator encodes.

In [ ]:
imp = model.feature_importances(importance_type='gain')
imp.loc[imp.max(axis=1).sort_values(ascending=False).index].head(12).round(3)

## 4. The CV ladder — random vs leave-station-out vs spatiotemporal

The single most important correctness lever is **how we split train/test**. A random k-fold
leaks spatial+temporal autocorrelation and inflates skill; leave-station-out scores only on
unseen physical sensors; spatiotemporal-blocked is the strictest. We run the ladder for PM2.5
with `run_cv_ladder` using the single-pollutant LightGBM factory (so the monotone constraint
holds inside every fold). The drop from `random_kfold` to the strict rungs **is** the leakage.

In [ ]:
from aqi_india.validation.cv import run_cv_ladder
from aqi_india.models.lightgbm_model import make_single_pollutant_factory

pol = 'pm25'
clean = matrix.dropna(subset=[pol, *feat_cols]).reset_index(drop=True)
factory = make_single_pollutant_factory(pol, params=params, feature_names=feat_cols)
ladder = run_cv_ladder(
    factory, clean[feat_cols], clean[pol].to_numpy(float),
    groups=clean['station_id'].to_numpy(), times=clean['time'].to_numpy(),
    lats=clean['lat'].to_numpy(), lons=clean['lon'].to_numpy(),
)
ladder

In [ ]:
from aqi_india.validation.plots import cv_ladder_bar
cv_ladder_bar(ladder, metric='r',
              title=f'CV ladder for {pol} — skill vs split rigour (leakage gap)')
import matplotlib.pyplot as plt; plt.show()

## 5. Out-of-fold predictions → validation figures + metrics table

For honest validation figures we use **leave-station-out out-of-fold** predictions (every
station predicted by a model that never saw it). We collect OOF pairs with `cross_validate`'s
splitter, then draw the Taylor diagram, the 1:1 hexbin and the per-station residual map, and
print the full metrics table (RMSE/MAE/MBE/R/R²/NMB/NME/IOA + RMA slope).

In [ ]:
from aqi_india.validation.cv import leave_station_out

X = clean[feat_cols]; y = clean[pol].to_numpy(float)
groups = clean['station_id'].to_numpy()
oof_true, oof_pred, oof_idx = [], [], []
for tr, te in leave_station_out(X, groups, n_splits=5):
    est = factory(); est.fit(X.iloc[tr], y[tr])
    oof_true.append(y[te]); oof_pred.append(np.asarray(est.predict(X.iloc[te])))
    oof_idx.append(te)
import numpy as np
oof_true = np.concatenate(oof_true); oof_pred = np.concatenate(oof_pred)
oof_idx = np.concatenate(oof_idx)
print('OOF pairs:', oof_true.size)

In [ ]:
from aqi_india.validation.metrics import metrics_table
tbl = metrics_table(oof_true, oof_pred)
pd.Series(tbl).round(3).to_frame(f'{pol} (leave-station-out OOF)')

In [ ]:
from aqi_india.validation.plots import one_to_one_hexbin, taylor_diagram, residual_map
import matplotlib.pyplot as plt

# 1:1 hexbin (predicted vs observed) with R / RMSE / MAE annotation.
one_to_one_hexbin(oof_true, oof_pred, units='ug/m3', title=f'{pol}: predicted vs observed (OOF)')
plt.show()

In [ ]:
# Taylor diagram — normalised std vs correlation for this model.
obs_std = float(np.std(oof_true))
stats = [{'name': f'LGBM {pol}', 'std': float(np.std(oof_pred)) / obs_std, 'r': tbl['r']}]
taylor_diagram(stats, ref_std=1.0, title=f'Taylor diagram — {pol} (normalised)')
plt.show()

In [ ]:
# Per-station residual (pred - obs) map over India.
resid = oof_pred - oof_true
stn = clean.iloc[oof_idx][['lat', 'lon']].reset_index(drop=True)
residual_map(stn, resid, title=f'{pol} residuals (pred - obs), leave-station-out OOF')
plt.show()

## Summary

End-to-end Objective-1 on synthetic data: gap-fill closed the cloud holes, the H3-keyed
physics feature matrix fed a per-pollutant LightGBM, and the **CV ladder made the
spatial/temporal leakage explicit** (random k-fold over-states skill vs leave-station-out and
spatiotemporal-blocked). The leave-station-out OOF metrics + Taylor/hexbin/residual figures
are the honest, reviewer-credible validation story. The same trained model feeds
`aqi_india.models.predict` to produce the gridded AQI map of notebook 02.